# 04 — Horizon topology and the photon shadow

**Spacetime Lab — Phase 4 concept + implementation notebook**

This notebook is the companion to `spacetime_lab.horizons`. It walks through the three notions of horizon (event, apparent, and 'computational'), demonstrates that all three coincide for stationary black holes, and ends with the headline result of the phase: a numerical reconstruction of the **Bardeen 1973 photon shadow** of a Kerr black hole — the same dark region that EHT measured for M87* in 2019 and Sgr A* in 2022.

**What you will learn**

1. The teleological definition of the event horizon and why it is hard to compute in dynamical spacetimes
2. The marginally outer-trapped surface (MOTS) and the apparent horizon as its outermost example
3. Why event = apparent for stationary BHs (Schwarzschild, Kerr) and why we can therefore use the algebraic shortcut $g^{rr} = 0$
4. The Schwarzschild critical impact parameter $b_\text{crit} = 3\sqrt{3}\,M$ and the photon sphere at $r = 3M$
5. The Bardeen 1973 parametric description of the Kerr shadow, with frame-dragging asymmetries that distort it from a circle
6. End-to-end cross-validation: every horizon finder rediscovers a textbook closed-form value, exercising Phase 1, Phase 3 and Phase 4 in one shot

**References**

- Hawking, *Black holes in general relativity*, CMP 25 152 (1972) — area theorem
- Bardeen, *Timelike and null geodesics in the Kerr metric*, in *Black Holes (Les Astres Occlus)*, 1973
- Bardeen, Press & Teukolsky, *ApJ* **178** 347 (1972) — ISCO formula
- [Stein, *Kerr Spherical Photon Orbits*](https://duetosymmetry.com/tool/kerr-circular-photon-orbits/)
- [Cunha et al, arXiv:1904.07710](https://arxiv.org/abs/1904.07710) — explicit alpha-beta formulas
- Event Horizon Telescope Collaboration, *First M87 Event Horizon Telescope Results*, ApJL **875** L1-L6 (2019)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.metrics import Schwarzschild, Kerr
from spacetime_lab.horizons import (
    find_event_horizon,
    find_isco_numerical,
    photon_shadow_kerr,
    spherical_photon_orbit_xi,
    spherical_photon_orbit_eta,
    kerr_critical_curve_xi_eta,
)

plt.rcParams['figure.figsize'] = (7, 4.5)

## 1. Three notions of horizon

The word 'horizon' in general relativity is overloaded. There are at least three inequivalent definitions:

**Event horizon** $\mathcal{H}^+$: the topological boundary of the past of future null infinity. In words: the boundary of 'the set of events from which a future-directed null geodesic can reach $\mathscr{I}^+$'. This is the textbook definition. It is **teleological**: to know whether a given event $p$ is on the horizon, you need to know the entire future of the spacetime — every null geodesic from $p$ has to be followed to infinity. There is no purely local test.

**Apparent horizon**: the outermost surface on a Cauchy slice where the expansion of the outgoing null normal vanishes:
$$\theta_{(\ell)} = 0.$$
Such surfaces are called **marginally outer-trapped surfaces (MOTS)**, and the apparent horizon is the outermost MOTS on each slice. It is **local in time** — only one slice is needed — but it depends on the slicing, unlike the event horizon.

**Computational horizon** (our shortcut): for stationary axisymmetric vacuum metrics, both event and apparent horizon coincide with the locus where $g^{rr}(r,\theta) = 0$. This is *algebraic*, fast, and numerically robust. It only works because of stationarity; for dynamical spacetimes (a collapsing star, two coalescing BHs) you would need a real MOTS finder, which is a 2D nonlinear PDE solver and out of scope for this phase.

## 2. Cross-validation: rediscover Schwarzschild and Kerr horizons

The whole point of Phase 4 is **rediscovery**: every closed-form value from Phases 1-3 should pop out of the numerical finder. If a finder ever disagrees with the analytical formula, we know exactly which layer of the stack is broken.

In [ ]:
# Schwarzschild: r_+ = 2M
for M in [0.5, 1.0, 2.5, 10.0]:
    sch = Schwarzschild(mass=M)
    rh = find_event_horizon(sch)
    print(f'Schwarzschild M={M:5.2f}:  numerical r_+ = {rh:.10f}   analytic 2M = {2*M:.10f}')

In [ ]:
# Kerr: r_+ = M + sqrt(M^2 - a^2)
M = 1.0
for a in [0.0, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]:
    k = Kerr(mass=M, spin=a)
    rh = find_event_horizon(k)
    expected = M + math.sqrt(M*M - a*a)
    err = abs(rh - expected)
    print(f'Kerr a={a:.2f}:  numerical r_+ = {rh:.10f}   analytic = {expected:.10f}   err = {err:.2e}')

## 3. Numerical ISCO via the effective potential

The Schwarzschild ISCO sits at the marginal-stability point of the equatorial radial effective potential — i.e., the radius where both $V_\text{eff}'(r;L) = 0$ and $V_\text{eff}''(r;L) = 0$ are satisfied simultaneously. This is two equations in two unknowns $(r, L)$ which we solve with a Newton-type root finder.

We do **not** use the closed-form value $r = 6M$. The point is to verify numerically that this comes out of the metric implementation.

In [ ]:
for M in [0.5, 1.0, 2.5, 10.0]:
    sch = Schwarzschild(mass=M)
    isco = find_isco_numerical(sch, r_guess=6*M, L_guess=3.5*M)
    print(f'Schwarzschild M={M:5.2f}:  numerical ISCO = {isco:.10f}   analytic 6M = {6*M:.10f}')

Linear scaling with mass holds to ~$10^{-9}$ in every case.

## 4. The Schwarzschild photon shadow

When you shine isotropic light on a Schwarzschild black hole and observe it from far away, you see a dark region. Photons with impact parameter
$$b > b_\text{crit} = 3\sqrt{3}\,M \approx 5.196\,M$$
escape; photons with $b < b_\text{crit}$ are captured; photons with exactly $b = b_\text{crit}$ asymptotically orbit on the photon sphere at $r = 3M$ before either escaping or falling in.

The shadow boundary in the observer's image plane is therefore a perfect circle of radius $3\sqrt{3}\,M$ — significantly **larger** than the Schwarzschild radius $r_+ = 2M$, by a factor of $1.5\sqrt{3} \approx 2.6$. This is the gravitational lensing magnification of the horizon.

In [ ]:
# Draw the Schwarzschild shadow analytically
M = 1.0
b_crit = 3.0 * math.sqrt(3.0)
phi = np.linspace(0, 2*np.pi, 200)
alpha_s = b_crit * np.cos(phi)
beta_s = b_crit * np.sin(phi)

fig, ax = plt.subplots(figsize=(7, 7))
ax.fill(alpha_s, beta_s, color='#222222', alpha=0.85, label=f'shadow (radius 3$\\sqrt{{3}}M$ ≈ {b_crit:.3f})')
# Show the horizon at r=2M for reference (the horizon is *inside* the shadow,
# magnified by gravitational lensing into the larger dark disk)
phi2 = np.linspace(0, 2*np.pi, 200)
ax.plot(2*M*np.cos(phi2), 2*M*np.sin(phi2), color='#a40000', lw=1.5, ls='--', label='event horizon (r=2M)')
ax.plot(3*M*np.cos(phi2), 3*M*np.sin(phi2), color='#0050a0', lw=1.5, ls=':', label='photon sphere (r=3M)')
ax.set_aspect('equal')
ax.set_xlim(-7, 7)
ax.set_ylim(-7, 7)
ax.set_xlabel(r'$\alpha / M$ (image plane horizontal)')
ax.set_ylabel(r'$\beta / M$ (image plane vertical)')
ax.set_title('Schwarzschild photon shadow')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Two important things to read off this plot:

1. **The shadow is not the horizon.** The dark disk has radius $3\sqrt{3}\,M \approx 5.2\,M$, but the actual event horizon (red dashed) is at $2\,M$. The factor of ~2.6 is **gravitational magnification**: distant photons that pass within $3\sqrt{3}\,M$ get bent into the photon sphere at $3\,M$ and ultimately fall in, even though they would naively (Euclideanly) miss the horizon by a wide margin.
2. **The photon sphere is at $3\,M$**, between the horizon and the shadow boundary. Photons grazing it asymptotically orbit there.

## 5. The Kerr photon shadow — Bardeen 1973

For a *rotating* black hole the shadow stops being a circle. Frame dragging asymmetrically captures prograde versus retrograde photons: prograde photons get sucked in more easily, so the prograde side of the shadow flattens; retrograde photons need a larger impact parameter to be captured, so the retrograde side bulges outward.

Bardeen's 1973 closed-form parametric description gives the shadow boundary as a curve $(\alpha(r_p), \beta(r_p))$ in the observer's image plane, parametrised by the radius $r_p$ of the corresponding spherical photon orbit. The conserved quantities of the spherical orbit are

$$\xi(r_p) = \frac{L_z}{E} = -\frac{r_p^3 - 3 M r_p^2 + a^2 r_p + a^2 M}{a (r_p - M)},$$

$$\eta(r_p) = \frac{\mathcal{Q}}{E^2} = -\frac{r_p^3 (r_p^3 - 6 M r_p^2 + 9 M^2 r_p - 4 a^2 M)}{a^2 (r_p - M)^2},$$

and the image-plane coordinates for an observer at polar inclination $\theta_o$ are

$$\alpha = -\xi \csc\theta_o, \qquad \beta = \pm\sqrt{\eta + a^2\cos^2\theta_o - \xi^2 \cot^2\theta_o}.$$

For the **equatorial observer** $\theta_o = \pi/2$ (the EHT geometry for both M87* and Sgr A*) these simplify enormously to $\alpha = -\xi$ and $\beta = \pm\sqrt{\eta}$.

In [ ]:
# Plot the shadow at five spin values
fig, ax = plt.subplots(figsize=(8, 7))

spins = [0.0, 0.3, 0.5, 0.9, 0.998]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(spins)))

# Schwarzschild shadow as analytic circle
phi = np.linspace(0, 2*np.pi, 200)
ax.plot(b_crit * np.cos(phi), b_crit * np.sin(phi), color=colors[0], lw=2.0, label='a = 0 (Schwarzschild)')

# Kerr shadows for non-zero spin
for spin, color in zip(spins[1:], colors[1:]):
    alpha, beta = photon_shadow_kerr(spin=spin, mass=1.0, n_points=300)
    ax.plot(alpha, beta, color=color, lw=2.0, label=f'a = {spin}')

ax.set_aspect('equal')
ax.set_xlabel(r'$\alpha / M$', fontsize=12)
ax.set_ylabel(r'$\beta / M$', fontsize=12)
ax.set_title('Bardeen 1973 photon shadow — equatorial observer', fontsize=13)
ax.set_xlim(-7, 7)
ax.set_ylim(-7, 7)
ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)
ax.grid(alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

Notice three features as the spin increases:

1. **The Schwarzschild circle stays as the reference** — at $a = 0$ the curve is a perfect circle of radius $3\sqrt{3}\,M$.
2. **The shadow shifts horizontally in $+\alpha$**. The center moves to the right, *not the projection of the singularity*. This is the famous frame-dragging shift.
3. **The left side flattens, the right side bulges.** Prograde photons (the ones moving with the BH rotation, on the left in this convention) get captured more easily; retrograde photons (right) need larger impact parameters to be caught. The shape becomes a 'D' or asymmetric bean.

At extremal Kerr ($a = M$) the asymmetry is maximal. **This is exactly the kind of distortion that EHT measurements probe** — the shape and size of the shadow constrain the spin and inclination of the source.

## 6. The 'EHT image' — extremal Kerr with a horizon overlay

Let's plot the most photogenic case: extremal Kerr ($a = 0.998M$, the astrophysical near-extremal limit), filled in dark, with the actual horizon and ergosphere drawn for context.

In [ ]:
M = 1.0
a = 0.998
k = Kerr(mass=M, spin=a)

alpha, beta = photon_shadow_kerr(spin=a, mass=M, n_points=400)

fig, ax = plt.subplots(figsize=(8, 8))

# Filled shadow
ax.fill(alpha, beta, color='#111111', alpha=0.85)
ax.plot(alpha, beta, color='#222222', lw=1.0)

# Horizon (actual r_+, projected radially)
rp = k.outer_horizon()
phi = np.linspace(0, 2*np.pi, 200)
ax.plot(rp*np.cos(phi), rp*np.sin(phi), color='#a40000', lw=1.5, ls='--', label=f'event horizon $r_+$ = {rp:.3f}')

# Equatorial photon spheres (prograde and retrograde)
ph_pro = k.photon_sphere_equatorial(prograde=True)
ph_retro = k.photon_sphere_equatorial(prograde=False)
ax.plot(ph_pro*np.cos(phi), ph_pro*np.sin(phi), color='#0050a0', lw=1.0, ls=':', label=f'prograde photon orbit ({ph_pro:.3f})')
ax.plot(ph_retro*np.cos(phi), ph_retro*np.sin(phi), color='#1a9a4a', lw=1.0, ls=':', label=f'retrograde photon orbit ({ph_retro:.3f})')

# Schwarzschild reference circle
ax.plot(b_crit*np.cos(phi), b_crit*np.sin(phi), color='#888888', lw=1.0, ls='-', alpha=0.6, label=f'Schwarzschild reference (radius 3$\\sqrt{{3}}M$)')

ax.set_aspect('equal')
ax.set_xlim(-8, 8)
ax.set_ylim(-8, 8)
ax.set_xlabel(r'$\alpha / M$', fontsize=12)
ax.set_ylabel(r'$\beta / M$', fontsize=12)
ax.set_title(f'Near-extremal Kerr photon shadow ($a = {a}\,M$, equatorial observer)', fontsize=13)
ax.axhline(0, color='k', lw=0.4)
ax.axvline(0, color='k', lw=0.4)
ax.grid(alpha=0.2)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

Notice that the dark shadow is offset from the actual event horizon. The horizon (red dashed) is much smaller than the shadow itself; the shadow is the *gravitationally lensed image* of the photon-capture region, not the horizon. The prograde and retrograde photon orbits (blue and green dotted) bracket the shadow at small radii.

## 7. Phase 4 gate checks

Before we move on to Phase 5 the following must hold. The assertions below fail loudly if anything regresses.

In [ ]:
# (1) find_event_horizon rediscovers Schwarzschild r_+ = 2M
for M in [0.5, 1.0, 2.5]:
    rh = find_event_horizon(Schwarzschild(mass=M))
    assert math.isclose(rh, 2.0 * M, abs_tol=1e-9)

# (2) find_event_horizon rediscovers Kerr r_+ = M + sqrt(M^2 - a^2)
for a in [0.0, 0.3, 0.5, 0.9, 0.99]:
    rh = find_event_horizon(Kerr(mass=1.0, spin=a))
    expected = 1.0 + math.sqrt(1.0 - a*a)
    assert math.isclose(rh, expected, abs_tol=1e-9)

# (3) Extremal Kerr horizon at r = M (parabolic-touch case)
rh = find_event_horizon(Kerr(mass=1.0, spin=1.0))
assert math.isclose(rh, 1.0, abs_tol=1e-6)

# (4) find_isco_numerical rediscovers Schwarzschild ISCO = 6M
for M in [0.5, 1.0, 2.5]:
    isco = find_isco_numerical(Schwarzschild(mass=M), r_guess=6*M, L_guess=3.5*M)
    assert math.isclose(isco, 6.0 * M, abs_tol=1e-7)

# (5) Schwarzschild critical impact parameter is 3 sqrt(3) M
alpha_sch, beta_sch = photon_shadow_kerr(spin=1e-6, mass=1.0, n_points=400)
radii = np.sqrt(alpha_sch**2 + beta_sch**2)
assert math.isclose(radii.mean(), 3.0 * math.sqrt(3.0), abs_tol=1e-2)

# (6) Frame-dragging shifts the Kerr shadow centroid in +alpha
for spin in [0.3, 0.5, 0.998]:
    a_arr, b_arr = photon_shadow_kerr(spin=spin, mass=1.0)
    assert a_arr.mean() > 0

# (7) Shadow scales linearly with M
a1, b1 = photon_shadow_kerr(spin=0.5, mass=1.0, n_points=100)
a2, b2 = photon_shadow_kerr(spin=1.0, mass=2.0, n_points=100)
ratio = np.sqrt(a2**2 + b2**2).mean() / np.sqrt(a1**2 + b1**2).mean()
assert math.isclose(ratio, 2.0, abs_tol=1e-6)

# (8) Eta vanishes at the equatorial photon-sphere endpoints
M, a = 1.0, 0.5
ph_pro = 2*M*(1 + math.cos((2/3)*math.acos(-a/M)))
eta_endpoint = spherical_photon_orbit_eta(ph_pro, M, a)
assert math.isclose(eta_endpoint, 0.0, abs_tol=1e-8)

print('ALL PHASE 4 GATE CHECKS PASSED')

---

## What's next — Phase 5 teaser

Phase 5 builds the **gravitational waves** module: quasinormal modes (QNMs) of Schwarzschild and Kerr black holes via the Leaver continued-fraction method, plus a basic ringdown waveform generator. The QNM frequencies $\omega_{lmn}(M, a)$ are the universal 'ringtones' of a perturbed black hole — every BH merger detected by LIGO/Virgo ends with a ringdown phase whose spectrum is given by these frequencies. They depend only on the final mass and spin, so they are how 'no-hair' is tested empirically.

Phase 5 will reuse Phase 1 (Schwarzschild), Phase 3 (Kerr), and Phase 4 (horizon — the Leaver method involves boundary conditions at the horizon). The cross-validation pattern stays the same: numerical QNM frequencies at $a = 0$ should match the well-known Schwarzschild values to machine precision.